## 1. Setup & Environment

In [1]:
import os
if not os.path.exists('./Student folder') and not os.path.exists('/content/drive/MyDrive/Student Folder'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass

In [2]:
import os
import time
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss, confusion_matrix

Backend: CPU (scikit-learn KDTree)


In [3]:
USE_GPU = False
try:
    from cuml.neighbors import KNeighborsClassifier as cuKNeighborsClassifier
    _probe = cuKNeighborsClassifier(n_neighbors=3, weights='distance')
    _probe.fit(np.zeros((6, 2), dtype=np.float32), np.array([0, 1, 0, 1, 0, 1]))
    USE_GPU = True
except Exception:
    USE_GPU = False

def to_numpy(arr):
    if hasattr(arr, 'to_numpy'):
        return arr.to_numpy()
    if hasattr(arr, 'get'):
        return arr.get()
    return np.asarray(arr)

def make_knn(n_neighbors):
    if USE_GPU:
        return cuKNeighborsClassifier(n_neighbors=n_neighbors, weights='distance')
    return KNeighborsClassifier(n_neighbors=n_neighbors, weights='distance', algorithm='kd_tree', leaf_size=40, n_jobs=-1)

print('Backend:', 'GPU (cuML)' if USE_GPU else 'CPU (scikit-learn KDTree)')

Train dir: ./Student folder/train | Exists: True
Test dir: ./Student folder/student_test | Exists: True


In [4]:
DATA_DIR = '/content/drive/MyDrive/Student Folder'
if not os.path.exists(DATA_DIR):
    DATA_DIR = './Student folder'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'student_test')
print('Train dir:', TRAIN_DIR, '| Exists:', os.path.exists(TRAIN_DIR))
print('Test dir:', TEST_DIR, '| Exists:', os.path.exists(TEST_DIR))

## 2. Data Preprocessing & Feature Engineering

In [5]:
def get_bearing(lon1, lat1, lon2, lat2):
    dlon = np.radians(lon2 - lon1)
    lat1_r = np.radians(lat1)
    lat2_r = np.radians(lat2)
    x = np.sin(dlon) * np.cos(lat2_r)
    y = np.cos(lat1_r) * np.sin(lat2_r) - np.sin(lat1_r) * np.cos(lat2_r) * np.cos(dlon)
    return np.degrees(np.arctan2(x, y)) % 360.0

def project_to_meters(lon, lat, lon0, lat0):
    r_earth = 6371000.0
    cos_lat = np.cos(np.radians(lat0))
    x = np.radians(lon - lon0) * r_earth * cos_lat
    y = np.radians(lat - lat0) * r_earth
    return np.column_stack([x, y])

def point_to_segment_batch(pts_xy, u_xy, v_xy):
    seg = v_xy - u_xy
    seg_len2 = np.sum(seg * seg, axis=2)
    diff = pts_xy[:, None, :] - u_xy
    dot = np.sum(diff * seg, axis=2)
    t = np.where(seg_len2 > 0.0, dot / np.maximum(seg_len2, 1e-12), 0.0)
    t = np.clip(t, 0.0, 1.0)
    proj = u_xy + t[:, :, None] * seg
    dists = np.linalg.norm(pts_xy[:, None, :] - proj, axis=2)
    return dists, t

def build_dataset(folder_path, k_candidates=10, k_pre=30, max_trips=None):
    X_list, y_list = [], []
    trip_group_list, obs_id_list, cand_arc_list = [], [], []
    arcs_by_trip = {}
    obs_offset = 0

    trips = sorted(os.listdir(folder_path))
    if max_trips:
        trips = trips[:max_trips]

    for trip_idx, trip in enumerate(trips):
        t_dir = os.path.join(folder_path, trip)
        if not os.path.isdir(t_dir):
            continue

        track = np.atleast_2d(np.loadtxt(os.path.join(t_dir, f'{trip}.track')))
        nodes = np.atleast_2d(np.loadtxt(os.path.join(t_dir, f'{trip}.nodes')))
        arcs = np.atleast_2d(np.loadtxt(os.path.join(t_dir, f'{trip}.arcs'), dtype=int))
        route_raw = np.loadtxt(os.path.join(t_dir, f'{trip}.route'), dtype=int)
        route_arr = np.atleast_1d(route_raw) if route_raw.size > 0 else np.empty(0, dtype=int)

        if track.size == 0 or nodes.size == 0 or arcs.size == 0:
            continue

        valid_track = np.isfinite(track).all(axis=1)
        track = track[valid_track]
        if len(track) == 0:
            continue

        if not np.isfinite(nodes).all():
            nodes = np.where(np.isfinite(nodes), nodes, np.nanmedian(nodes, axis=0))

        if len(track) > 1 and np.any(np.diff(track[:, 2]) < 0):
            track = track[np.argsort(track[:, 2])]

        arcs_by_trip[trip_idx] = arcs
        route_set = set(route_arr.tolist())

        lon0 = float(np.mean(track[:, 0]))
        lat0 = float(np.mean(track[:, 1]))
        track_xy = project_to_meters(track[:, 0], track[:, 1], lon0, lat0)
        nodes_xy = project_to_meters(nodes[:, 0], nodes[:, 1], lon0, lat0)

        u_xy = nodes_xy[arcs[:, 0]]
        v_xy = nodes_xy[arcs[:, 1]]
        midpoints_xy = 0.5 * (u_xy + v_xy)
        arc_lens = np.linalg.norm(v_xy - u_xy, axis=1)

        u_ll = nodes[arcs[:, 0]]
        v_ll = nodes[arcs[:, 1]]
        arc_bearings = get_bearing(u_ll[:, 0], u_ll[:, 1], v_ll[:, 0], v_ll[:, 1])

        n_pts = len(track)
        if n_pts > 1:
            dt = np.maximum(np.diff(track[:, 2]), 1e-3)
            step_dist = np.linalg.norm(np.diff(track_xy, axis=0), axis=1)
            speeds_step = np.clip(step_dist / dt, 0.0, 60.0)
            bearings_step = get_bearing(track[:-1, 0], track[:-1, 1], track[1:, 0], track[1:, 1])
            gps_speeds = np.append(speeds_step, speeds_step[-1])
            gps_bearings = np.append(bearings_step, bearings_step[-1])
        else:
            gps_speeds = np.array([0.0])
            gps_bearings = np.array([0.0])

        k_use = min(k_candidates, len(arcs))
        k_fetch = min(max(k_pre, k_use), len(arcs))
        tree = cKDTree(midpoints_xy)
        mid_dists_all, cand_ids_all = tree.query(track_xy, k=k_fetch)
        mid_dists_all = np.asarray(mid_dists_all).reshape(n_pts, k_fetch)
        cand_ids_all = np.asarray(cand_ids_all, dtype=int).reshape(n_pts, k_fetch)

        seg_dists_all, t_proj_all = point_to_segment_batch(track_xy, u_xy[cand_ids_all], v_xy[cand_ids_all])
        order = np.argsort(seg_dists_all, axis=1)[:, :k_use]
        row_idx = np.arange(n_pts)[:, None]

        cand_ids = cand_ids_all[row_idx, order]
        seg_dists = seg_dists_all[row_idx, order]
        mid_dists = mid_dists_all[row_idx, order]
        t_proj = t_proj_all[row_idx, order]
        rel_dists = seg_dists - seg_dists[:, :1]
        cand_arc_lens = arc_lens[cand_ids]

        b_diff = np.abs(gps_bearings[:, None] - arc_bearings[cand_ids])
        b_diff = np.minimum(b_diff, 360.0 - b_diff)
        cand_speeds = np.repeat(gps_speeds[:, None], k_use, axis=1)
        t_center_dist = np.abs(t_proj - 0.5)

        feats = np.stack(
            [seg_dists, mid_dists, rel_dists, cand_arc_lens, b_diff, cand_speeds, t_center_dist],
            axis=2
        ).reshape(-1, 7).astype(np.float32)

        labels = np.isin(cand_ids, list(route_set)).astype(np.int32).ravel()
        obs_ids = obs_offset + np.repeat(np.arange(n_pts, dtype=np.int64), k_use)
        trip_ids = np.full(len(labels), trip_idx, dtype=np.int32)

        X_list.append(feats)
        y_list.append(labels)
        trip_group_list.append(trip_ids)
        obs_id_list.append(obs_ids)
        cand_arc_list.append(cand_ids.ravel().astype(np.int32))
        obs_offset += n_pts

    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    meta = {
        'trip_ids': np.concatenate(trip_group_list),
        'obs_ids': np.concatenate(obs_id_list),
        'cand_arcs': np.concatenate(cand_arc_list),
        'n_obs': obs_offset,
        'n_trips': len(arcs_by_trip)
    }
    return X, y, meta, arcs_by_trip

In [6]:
def evaluate_predictions(y_true, y_pred, y_prob, meta, arcs_by_trip):
    obs_ids = meta['obs_ids']
    trip_ids = meta['trip_ids']
    cand_arcs = meta['cand_arcs']

    obs_starts = np.flatnonzero(np.r_[True, obs_ids[1:] != obs_ids[:-1]])
    obs_ends = np.r_[obs_starts[1:], len(obs_ids)]

    top1_hits = 0
    misses = 0
    pred_seqs = {}
    for s, e in zip(obs_starts, obs_ends):
        best_i = s + int(np.argmax(y_prob[s:e]))
        if y_true[best_i] == 1:
            top1_hits += 1
        if np.max(y_true[s:e]) == 0:
            misses += 1
        tid = int(trip_ids[s])
        pred_seqs.setdefault(tid, []).append(int(cand_arcs[best_i]))

    cont_hits = 0
    cont_total = 0
    for tid, seq in pred_seqs.items():
        arcs = arcs_by_trip[tid]
        for a1, a2 in zip(seq[:-1], seq[1:]):
            cont_total += 1
            if a1 == a2 or arcs[a1, 1] == arcs[a2, 0]:
                cont_hits += 1

    return {
        'candidate_acc': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'log_loss': log_loss(y_true, y_prob, labels=[0, 1]),
        'top1_acc': top1_hits / len(obs_starts),
        'miss_rate': misses / len(obs_starts),
        'route_continuity': cont_hits / max(cont_total, 1)
    }

## 3. Dataset Construction & Inspection

In [7]:
t0 = time.time()
K_CANDIDATES = 10
X_train, y_train, train_meta, train_arcs = build_dataset(TRAIN_DIR, k_candidates=K_CANDIDATES, k_pre=30)
X_test, y_test, test_meta, test_arcs = build_dataset(TEST_DIR, k_candidates=K_CANDIDATES, k_pre=30)
print('Feature build time (s):', round(time.time() - t0, 2))
print('Train:', train_meta['n_trips'], 'trips |', train_meta['n_obs'], 'GPS pts | X:', X_train.shape, '| Pos ratio:', round(float(y_train.mean()), 4))
print('Test: ', test_meta['n_trips'], 'trips |', test_meta['n_obs'], 'GPS pts | X:', X_test.shape, '| Pos ratio:', round(float(y_test.mean()), 4))

Feature build time (s): 10.17
Train: 80 trips | 208417 GPS pts | X: (2084170, 7) | Pos ratio: 0.3296
Test:  10 trips | 19160 GPS pts | X: (191600, 7) | Pos ratio: 0.3851


## 4. Group-Aware CV & KNN Neighbor Selection

In [8]:
gkf = GroupKFold(n_splits=5)
train_groups = train_meta['trip_ids']
k_grid = [7, 15, 25]
cv_results = []

for k_val in k_grid:
    fold_f1, fold_loss, fold_top1 = [], [], []
    t_cv = time.time()
    for tr_idx, va_idx in gkf.split(X_train, y_train, train_groups):
        scaler_cv = StandardScaler()
        X_tr = scaler_cv.fit_transform(X_train[tr_idx]).astype(np.float32)
        X_va = scaler_cv.transform(X_train[va_idx]).astype(np.float32)
        model_cv = make_knn(k_val)
        model_cv.fit(X_tr, y_train[tr_idx])
        proba_va = to_numpy(model_cv.predict_proba(X_va))
        prob_va = proba_va[:, 1]
        pred_va = np.argmax(proba_va, axis=1)

        va_meta = {
            'obs_ids': train_meta['obs_ids'][va_idx],
            'trip_ids': train_meta['trip_ids'][va_idx],
            'cand_arcs': train_meta['cand_arcs'][va_idx]
        }
        m_va = evaluate_predictions(y_train[va_idx], pred_va, prob_va, va_meta, train_arcs)
        fold_f1.append(m_va['macro_f1'])
        fold_loss.append(m_va['log_loss'])
        fold_top1.append(m_va['top1_acc'])

    row = {
        'n_neighbors': k_val,
        'cv_top1_acc': round(float(np.mean(fold_top1)), 4),
        'cv_macro_f1': round(float(np.mean(fold_f1)), 4),
        'cv_log_loss': round(float(np.mean(fold_loss)), 4),
        'cv_time_s': round(time.time() - t_cv, 2)
    }
    cv_results.append(row)
    print(row)

cv_df = pd.DataFrame(cv_results)
best_k = int(cv_df.loc[cv_df['cv_top1_acc'].idxmax(), 'n_neighbors'])
print('Selected n_neighbors:', best_k)

{'n_neighbors': 7, 'cv_top1_acc': 0.6729, 'cv_macro_f1': 0.737, 'cv_log_loss': 1.6898, 'cv_time_s': 227.82}
{'n_neighbors': 15, 'cv_top1_acc': 0.679, 'cv_macro_f1': 0.7465, 'cv_log_loss': 0.9727, 'cv_time_s': 296.25}
{'n_neighbors': 25, 'cv_top1_acc': 0.6824, 'cv_macro_f1': 0.7496, 'cv_log_loss': 0.7383, 'cv_time_s': 319.38}
Selected n_neighbors: 25


## 5. Final KNN Training & Student-Test Evaluation

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

knn = make_knn(best_k)
t_fit = time.time()
knn.fit(X_train_scaled, y_train)
train_time = time.time() - t_fit

t_inf = time.time()
proba_test = to_numpy(knn.predict_proba(X_test_scaled))
y_prob = proba_test[:, 1]
y_pred = np.argmax(proba_test, axis=1)
inf_time = time.time() - t_inf

metrics = evaluate_predictions(y_test, y_pred, y_prob, test_meta, test_arcs)
cm = confusion_matrix(y_test, y_pred)

print('n_neighbors:', best_k, '| k_candidates:', K_CANDIDATES)
print('Training Time (s):', round(train_time, 4))
print('Inference Time (s):', round(inf_time, 4))
print('Candidate Accuracy:', round(metrics['candidate_acc'], 4))
print('Precision:', round(metrics['precision'], 4))
print('Recall:', round(metrics['recall'], 4))
print('Macro F1:', round(metrics['macro_f1'], 4))
print('Log Loss:', round(metrics['log_loss'], 4))
print('Observation-level Top-1 Accuracy:', round(metrics['top1_acc'], 4))
print('Candidate Miss Rate:', round(metrics['miss_rate'], 4))
print('Route Continuity:', round(metrics['route_continuity'], 4))
print('Confusion Matrix:\n', cm)

n_neighbors: 25 | k_candidates: 10
Training Time (s): 8.0845
Inference Time (s): 22.2266
Candidate Accuracy: 0.7743
Precision: 0.7473
Recall: 0.6254
Macro F1: 0.7531
Log Loss: 1.0281
Observation-level Top-1 Accuracy: 0.8077
Candidate Miss Rate: 0.004
Route Continuity: 0.5941
Confusion Matrix:
 [[102206  15607]
 [ 27642  46145]]
